# 🚀 ED Analytics — Entrenamiento YOLOv8s en Google Colab (GPU)

**Tiempo estimado: 25-40 minutos con GPU T4 gratuita**

### Pasos:
1. Ejecuta cada celda en orden (Shift+Enter)
2. En la celda de 'Subir dataset', sube el ZIP con tus imágenes
3. Al terminar, descarga `best.pt` y ponlo en `c:\apped\train_yolo\runs\detect\train\weights\best.pt`

In [ ]:
# 1. Verificar GPU disponible
import torch
print(f'GPU disponible: {torch.cuda.is_available()}')
print(f'Dispositivo: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (cambia a GPU en Runtime > Cambiar tipo de entorno de ejecución"}')
!nvidia-smi

In [ ]:
# 2. Instalar dependencias
!pip install ultralytics roboflow -q
print('✅ Dependencias instaladas')

In [ ]:
# 3. Opción A: Descargar dataset desde Roboflow (1000+ imágenes públicas)
from roboflow import Roboflow

API_KEY = '1FzYekbvWJQGv1HYQRvE'  # Tu API key
rf = Roboflow(api_key=API_KEY)
project = rf.workspace('roboflow-jvuqo').project('football-players-detection-3zvbc')
dataset = project.version(1).download('yolov8', location='/content/dataset')
print(f'Dataset en: {dataset.location}')

import os
for split in ['train', 'valid', 'test']:
    path = f'/content/dataset/{split}/images'
    if os.path.exists(path):
        n = len(os.listdir(path))
        print(f'  {split}: {n} imágenes')

DATASET_YAML = '/content/dataset/data.yaml'

In [ ]:
# 3. Opción B: Subir tu propio dataset (ZIP con estructura dataset/images/train + dataset/labels/train)
# Descomenta esta celda si prefieres usar TU dataset local

# from google.colab import files
# uploaded = files.upload()  # Sube el ZIP
# !unzip -q *.zip -d /content/
# DATASET_YAML = '/content/train_yolo/data.yaml'

In [ ]:
# 4. Ver el data.yaml para confirmar estrutura
with open(DATASET_YAML) as f:
    print(f.read())

In [ ]:
# 5. ENTRENAMIENTO — YOLOv8s con dataset de fútbol
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # Small: mejor balance precision/velocidad

results = model.train(
    data=DATASET_YAML,
    epochs=50,          # Suficiente con 1000+ imágenes
    imgsz=640,
    batch=16,           # GPU T4 soporta batch 16 perfectamente
    workers=4,
    project='/content/runs',
    name='ed_football',
    patience=15,
    save=True,
    plots=True,
    # Augmentación moderada (dataset ya variado)
    augment=True,
    fliplr=0.5,
    mosaic=0.8,
    hsv_s=0.4,
    hsv_v=0.3,
    degrees=5,
)

print(f'\n✅ Entrenamiento completado!')
print(f'mAP50: {results.results_dict.get("metrics/mAP50(B)", 0):.3f}')

In [ ]:
# 6. Validación del modelo
best_model = YOLO('/content/runs/ed_football/weights/best.pt')
val_results = best_model.val(data=DATASET_YAML)
print(f'mAP50: {val_results.box.map50:.3f}')
print(f'mAP50-95: {val_results.box.map:.3f}')
print(f'Clases: {best_model.names}')

In [ ]:
# 7. Descargar el modelo entrenado
from google.colab import files

best_path = '/content/runs/ed_football/weights/best.pt'
import os
size_mb = os.path.getsize(best_path) / 1024 / 1024
print(f'Descargando best.pt ({size_mb:.1f} MB)...')
print('\n⬇️ Guárdalo en: c:\\apped\\train_yolo\\runs\\detect\\train\\weights\\best.pt')
files.download(best_path)